# Cell 1 — Import library

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# Cell 2 — Tentukan folder project

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Output tables dir:", OUTPUT_TABLES_DIR)

Project root: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL
Processed dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed
Output tables dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables


# Cell 3 — Baca dataset fitur process mining

In [3]:
FEATURE_PATH = PROCESSED_DIR / "08_features_process_mining_per_student.csv"

feature_df = pd.read_csv(FEATURE_PATH)

print("Ukuran feature_df:", feature_df.shape)
display(feature_df.head())

print("\nDistribusi label_performance:")
display(feature_df["label_performance"].value_counts())

Ukuran feature_df: (89, 65)


,case_id,nim,nama,kelas,nilai_total,nilai_indeks,label_performance,total_events_full,unique_activity_full,total_events_compact,...,tr_submitted_to_quiz_module,variant_id,variant_frequency,variant_sequence,trace_is_fit,trace_fitness,missing_tokens,remaining_tokens,consumed_tokens,produced_tokens
0,ADITYA PUTRA PERMANA,103012530065,ADITYA PUTRA PERMANA,IF-49-02,73.01,B,Sedang,2413,17,1979,...,22,1,1,Course Viewed -> Material Viewed -> Course Vie...,1,1.0,0,0,3866,3866
1,ADY RAHMAN WICAKSONO,103012500077,ADY RAHMAN WICAKSONO,IF-49-03,60.04,BC,Sedang,1460,17,1086,...,20,2,1,Course Viewed -> Material Viewed -> Assignment...,1,1.0,0,0,2096,2096
2,AGUNG KALEB SASAUW,103012500211,AGUNG KALEB SASAUW,IF-49-03,75.04,AB,Tinggi,1612,17,1266,...,26,3,1,Course Viewed -> Material Viewed -> Course Vie...,1,1.0,0,0,2488,2488
3,ALFITO MAULANA,103012530032,ALFITO MAULANA,IF-49-03,67.24,B,Sedang,1181,17,948,...,23,4,1,Quiz Module Viewed -> Material Viewed -> Quiz ...,1,1.0,0,0,1856,1856
4,ALFONSUS LIGUORI DANGO,103012580046,ALFONSUS LIGUORI DANGO,IF-49-02,40.28,D,Rendah,1000,12,844,...,15,5,1,Course Viewed -> Material Viewed -> Course Vie...,1,1.0,0,0,1684,1684



Distribusi label_performance:


label_performance
Sedang    54
Tinggi    20
Rendah    15
Name: count, dtype: int64

# Cell 4 — Cek kolom dataset

In [4]:
print("Daftar kolom:")
for i, col in enumerate(feature_df.columns, start=1):
    print(i, col)

Daftar kolom:
1 case_id
2 nim
3 nama
4 kelas
5 nilai_total
6 nilai_indeks
7 label_performance
8 total_events_full
9 unique_activity_full
10 total_events_compact
11 unique_activity_compact
12 compression_ratio
13 first_timestamp
14 last_timestamp
15 active_days
16 activity_span_days
17 avg_events_per_active_day
18 freq_course
19 freq_material
20 freq_quiz
21 freq_assignment
22 freq_forum
23 ratio_course
24 ratio_material
25 ratio_quiz
26 ratio_assignment
27 ratio_forum
28 act_course_viewed
29 act_material_viewed
30 act_quiz_module_viewed
31 act_quiz_access_prevented
32 act_quiz_started
33 act_quiz_viewed
34 act_quiz_updated
35 act_quiz_auto_saved
36 act_quiz_summary_viewed
37 act_quiz_submitted
38 act_assignment_viewed
39 act_assignment_status_viewed
40 act_forum_viewed
41 total_transition
42 unique_transition_count
43 tr_course_to_material
44 tr_course_to_quiz_module
45 tr_material_to_course
46 tr_quiz_module_to_course
47 tr_quiz_module_to_quiz_viewed
48 tr_quiz_module_to_quiz_started


# Cell 5 — Tentukan target dan kolom yang tidak dipakai sebagai fitur

In [5]:
target_col = "label_performance"

metadata_cols = [
    "case_id",
    "nim",
    "nama",
    "kelas",
    "nilai_total",
    "nilai_indeks",
    "label_performance"
]

text_or_sequence_cols = [
    "variant_sequence",
    "first_timestamp",
    "last_timestamp"
]

conformance_constant_or_technical_cols = [
    "trace_is_fit",
    "trace_fitness",
    "missing_tokens",
    "remaining_tokens",
    "consumed_tokens",
    "produced_tokens"
]

variant_cols_to_exclude = [
    "variant_id",
    "variant_frequency"
]

excluded_cols = (
    metadata_cols
    + text_or_sequence_cols
    + conformance_constant_or_technical_cols
    + variant_cols_to_exclude
)

print("Jumlah kolom yang dikecualikan:", len(excluded_cols))
print(excluded_cols)

Jumlah kolom yang dikecualikan: 18
['case_id', 'nim', 'nama', 'kelas', 'nilai_total', 'nilai_indeks', 'label_performance', 'variant_sequence', 'first_timestamp', 'last_timestamp', 'trace_is_fit', 'trace_fitness', 'missing_tokens', 'remaining_tokens', 'consumed_tokens', 'produced_tokens', 'variant_id', 'variant_frequency']


# Cell 6 — Pilih fitur numerik kandidat

In [6]:
numeric_cols = feature_df.select_dtypes(include=["number"]).columns.tolist()

candidate_features = [
    col for col in numeric_cols
    if col not in excluded_cols
]

print("Jumlah fitur numerik kandidat:", len(candidate_features))
print(candidate_features)

Jumlah fitur numerik kandidat: 47
['total_events_full', 'unique_activity_full', 'total_events_compact', 'unique_activity_compact', 'compression_ratio', 'active_days', 'activity_span_days', 'avg_events_per_active_day', 'freq_course', 'freq_material', 'freq_quiz', 'freq_assignment', 'freq_forum', 'ratio_course', 'ratio_material', 'ratio_quiz', 'ratio_assignment', 'ratio_forum', 'act_course_viewed', 'act_material_viewed', 'act_quiz_module_viewed', 'act_quiz_access_prevented', 'act_quiz_started', 'act_quiz_viewed', 'act_quiz_updated', 'act_quiz_auto_saved', 'act_quiz_summary_viewed', 'act_quiz_submitted', 'act_assignment_viewed', 'act_assignment_status_viewed', 'act_forum_viewed', 'total_transition', 'unique_transition_count', 'tr_course_to_material', 'tr_course_to_quiz_module', 'tr_material_to_course', 'tr_quiz_module_to_course', 'tr_quiz_module_to_quiz_viewed', 'tr_quiz_module_to_quiz_started', 'tr_quiz_started_to_quiz_viewed', 'tr_quiz_viewed_to_quiz_updated', 'tr_quiz_updated_to_quiz_v

# Cell 7 — Buang fitur konstan

In [7]:
constant_features = [
    col for col in candidate_features
    if feature_df[col].nunique() <= 1
]

selected_features = [
    col for col in candidate_features
    if col not in constant_features
]

print("Jumlah fitur konstan:", len(constant_features))
print(constant_features)

print("\nJumlah selected_features:", len(selected_features))
print(selected_features)

Jumlah fitur konstan: 0
[]

Jumlah selected_features: 47
['total_events_full', 'unique_activity_full', 'total_events_compact', 'unique_activity_compact', 'compression_ratio', 'active_days', 'activity_span_days', 'avg_events_per_active_day', 'freq_course', 'freq_material', 'freq_quiz', 'freq_assignment', 'freq_forum', 'ratio_course', 'ratio_material', 'ratio_quiz', 'ratio_assignment', 'ratio_forum', 'act_course_viewed', 'act_material_viewed', 'act_quiz_module_viewed', 'act_quiz_access_prevented', 'act_quiz_started', 'act_quiz_viewed', 'act_quiz_updated', 'act_quiz_auto_saved', 'act_quiz_summary_viewed', 'act_quiz_submitted', 'act_assignment_viewed', 'act_assignment_status_viewed', 'act_forum_viewed', 'total_transition', 'unique_transition_count', 'tr_course_to_material', 'tr_course_to_quiz_module', 'tr_material_to_course', 'tr_quiz_module_to_course', 'tr_quiz_module_to_quiz_viewed', 'tr_quiz_module_to_quiz_started', 'tr_quiz_started_to_quiz_viewed', 'tr_quiz_viewed_to_quiz_updated', 'tr

# Cell 8 — Buat dataset ML utama

In [8]:
ml_dataset = feature_df[selected_features + [target_col]].copy()

print("Ukuran ml_dataset:", ml_dataset.shape)
display(ml_dataset.head())

print("\nMissing value:")
display(ml_dataset.isna().sum()[ml_dataset.isna().sum() > 0])

print("\nDistribusi target:")
display(ml_dataset[target_col].value_counts())

Ukuran ml_dataset: (89, 48)


,total_events_full,unique_activity_full,total_events_compact,unique_activity_compact,compression_ratio,active_days,activity_span_days,avg_events_per_active_day,freq_course,freq_material,...,tr_quiz_module_to_quiz_started,tr_quiz_started_to_quiz_viewed,tr_quiz_viewed_to_quiz_updated,tr_quiz_updated_to_quiz_viewed,tr_quiz_updated_to_auto_saved,tr_auto_saved_to_quiz_viewed,tr_quiz_updated_to_summary,tr_summary_to_submitted,tr_submitted_to_quiz_module,label_performance
0,2413,17,1979,17,0.8201,63,121,38.30,166,122,...,7,16,670,611,69,74,19,14,22,Sedang
1,1460,17,1086,17,0.7438,44,118,33.18,159,97,...,7,18,322,295,16,16,30,16,20,Sedang
2,1612,17,1266,17,0.7854,47,117,34.30,95,141,...,11,18,376,345,29,35,27,23,26,Tinggi
3,1181,17,948,17,0.8027,37,113,31.92,74,54,...,11,18,298,280,10,7,21,15,23,Sedang
4,1000,12,844,12,0.8440,25,116,40.00,63,33,...,5,15,292,253,27,29,16,10,15,Rendah



Missing value:


Series([], dtype: int64)


Distribusi target:


label_performance
Sedang    54
Tinggi    20
Rendah    15
Name: count, dtype: int64

# Cell 9 — Buat mapping label numerik

In [9]:
label_mapping = {
    "Rendah": 0,
    "Sedang": 1,
    "Tinggi": 2
}

ml_dataset["label_encoded"] = ml_dataset[target_col].map(label_mapping)

display(ml_dataset[[target_col, "label_encoded"]].drop_duplicates().sort_values("label_encoded"))

,label_performance,label_encoded
4,Rendah,0
0,Sedang,1
2,Tinggi,2


# Cell 10 — Split train dan test stratified

In [10]:
X = ml_dataset[selected_features].copy()
y = ml_dataset[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nDistribusi y_train:")
display(y_train.value_counts())

print("\nDistribusi y_test:")
display(y_test.value_counts())

X_train: (71, 47)
X_test: (18, 47)
y_train: (71,)
y_test: (18,)

Distribusi y_train:


label_performance
Sedang    43
Tinggi    16
Rendah    12
Name: count, dtype: int64


Distribusi y_test:


label_performance
Sedang    11
Tinggi     4
Rendah     3
Name: count, dtype: int64

# Cell 11 — Gabungkan ulang train/test dengan target

In [11]:
train_df = X_train.copy()
train_df[target_col] = y_train.values
train_df["label_encoded"] = train_df[target_col].map(label_mapping)

test_df = X_test.copy()
test_df[target_col] = y_test.values
test_df["label_encoded"] = test_df[target_col].map(label_mapping)

display(train_df.head())
display(test_df.head())

,total_events_full,unique_activity_full,total_events_compact,unique_activity_compact,compression_ratio,active_days,activity_span_days,avg_events_per_active_day,freq_course,freq_material,...,tr_quiz_started_to_quiz_viewed,tr_quiz_viewed_to_quiz_updated,tr_quiz_updated_to_quiz_viewed,tr_quiz_updated_to_auto_saved,tr_auto_saved_to_quiz_viewed,tr_quiz_updated_to_summary,tr_summary_to_submitted,tr_submitted_to_quiz_module,label_performance,label_encoded
50,1606,18,1203,18,0.7491,30,114,53.53,53,137,...,17,424,393,26,20,24,16,23,Sedang,1
49,822,17,671,17,0.8163,34,113,24.18,40,33,...,14,219,194,17,15,15,11,15,Sedang,1
24,1487,17,1164,17,0.7828,49,116,30.35,80,72,...,15,392,368,18,18,21,16,20,Sedang,1
68,1323,16,1055,16,0.7974,39,114,33.92,67,51,...,26,312,273,27,32,30,23,26,Sedang,1
22,1850,17,1458,17,0.7881,50,114,37.00,86,101,...,15,468,429,40,45,24,18,21,Tinggi,2


,total_events_full,unique_activity_full,total_events_compact,unique_activity_compact,compression_ratio,active_days,activity_span_days,avg_events_per_active_day,freq_course,freq_material,...,tr_quiz_started_to_quiz_viewed,tr_quiz_viewed_to_quiz_updated,tr_quiz_updated_to_quiz_viewed,tr_quiz_updated_to_auto_saved,tr_auto_saved_to_quiz_viewed,tr_quiz_updated_to_summary,tr_summary_to_submitted,tr_submitted_to_quiz_module,label_performance,label_encoded
40,1515,17,1222,17,0.8066,48,117,31.56,67,59,...,15,400,370,39,37,25,19,25,Tinggi,2
78,3154,17,2574,17,0.8161,58,118,54.38,173,166,...,24,922,861,64,69,29,15,26,Sedang,1
30,1107,17,875,17,0.7904,45,117,24.60,63,50,...,18,286,271,6,4,23,20,25,Rendah,0
47,229,10,189,10,0.8253,12,112,19.08,14,4,...,2,65,60,2,2,5,3,3,Rendah,0
70,1216,17,993,17,0.8166,42,118,28.95,78,58,...,15,302,270,37,35,22,17,21,Sedang,1


# Cell 12 — Buat ringkasan distribusi kelas train/test

In [12]:
class_distribution = pd.DataFrame({
    "dataset": ["full", "train", "test"],
    "jumlah_data": [
        len(ml_dataset),
        len(train_df),
        len(test_df)
    ],
    "rendah": [
        (ml_dataset[target_col] == "Rendah").sum(),
        (train_df[target_col] == "Rendah").sum(),
        (test_df[target_col] == "Rendah").sum()
    ],
    "sedang": [
        (ml_dataset[target_col] == "Sedang").sum(),
        (train_df[target_col] == "Sedang").sum(),
        (test_df[target_col] == "Sedang").sum()
    ],
    "tinggi": [
        (ml_dataset[target_col] == "Tinggi").sum(),
        (train_df[target_col] == "Tinggi").sum(),
        (test_df[target_col] == "Tinggi").sum()
    ]
})

display(class_distribution)

CLASS_DISTRIBUTION_PATH = OUTPUT_TABLES_DIR / "10_class_distribution_train_test.csv"
class_distribution.to_csv(CLASS_DISTRIBUTION_PATH, index=False)

print("Distribusi kelas berhasil disimpan ke:")
print(CLASS_DISTRIBUTION_PATH)

,dataset,jumlah_data,rendah,sedang,tinggi
0,full,89,15,54,20
1,train,71,12,43,16
2,test,18,3,11,4


Distribusi kelas berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\10_class_distribution_train_test.csv


# Cell 13 — Buat beberapa feature set

In [13]:
feature_sets = {}

feature_sets["all_process_features"] = selected_features

feature_sets["core_behavior_features"] = [
    col for col in [
        "total_events_full",
        "total_events_compact",
        "compression_ratio",
        "active_days",
        "activity_span_days",
        "avg_events_per_active_day",
        "unique_activity_full",
        "unique_activity_compact",
        "total_transition",
        "unique_transition_count",
        "freq_course",
        "freq_material",
        "freq_quiz",
        "freq_assignment",
        "freq_forum",
        "ratio_course",
        "ratio_material",
        "ratio_quiz",
        "ratio_assignment",
        "ratio_forum"
    ]
    if col in selected_features
]

feature_sets["activity_specific_features"] = [
    col for col in selected_features
    if col.startswith("act_")
]

feature_sets["transition_specific_features"] = [
    col for col in selected_features
    if col.startswith("tr_")
]

feature_sets["frequency_and_ratio_features"] = [
    col for col in selected_features
    if col.startswith("freq_") or col.startswith("ratio_")
]

feature_set_summary = pd.DataFrame([
    {
        "feature_set": name,
        "jumlah_fitur": len(cols),
        "fitur": ", ".join(cols)
    }
    for name, cols in feature_sets.items()
])

display(feature_set_summary)

,feature_set,jumlah_fitur,fitur
0,all_process_features,47,"total_events_full, unique_activity_full, total..."
1,core_behavior_features,20,"total_events_full, total_events_compact, compr..."
2,activity_specific_features,13,"act_course_viewed, act_material_viewed, act_qu..."
3,transition_specific_features,14,"tr_course_to_material, tr_course_to_quiz_modul..."
4,frequency_and_ratio_features,10,"freq_course, freq_material, freq_quiz, freq_as..."


# Cell 14 — Simpan feature set ke JSON dan CSV

In [14]:
FEATURE_SET_JSON_PATH = PROCESSED_DIR / "10_feature_sets.json"
FEATURE_SET_SUMMARY_PATH = OUTPUT_TABLES_DIR / "10_feature_set_summary.csv"

with open(FEATURE_SET_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_sets, f, indent=4, ensure_ascii=False)

feature_set_summary.to_csv(FEATURE_SET_SUMMARY_PATH, index=False)

print("Feature sets JSON disimpan ke:")
print(FEATURE_SET_JSON_PATH)

print("\nFeature set summary disimpan ke:")
print(FEATURE_SET_SUMMARY_PATH)

Feature sets JSON disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\10_feature_sets.json

Feature set summary disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\10_feature_set_summary.csv


# Cell 15 — Simpan dataset ML

In [15]:
ML_DATASET_PATH = PROCESSED_DIR / "10_ml_dataset.csv"
TRAIN_DATASET_PATH = PROCESSED_DIR / "10_train_dataset.csv"
TEST_DATASET_PATH = PROCESSED_DIR / "10_test_dataset.csv"

ml_dataset.to_csv(ML_DATASET_PATH, index=False)
train_df.to_csv(TRAIN_DATASET_PATH, index=False)
test_df.to_csv(TEST_DATASET_PATH, index=False)

print("Dataset ML penuh disimpan ke:")
print(ML_DATASET_PATH)

print("\nTrain dataset disimpan ke:")
print(TRAIN_DATASET_PATH)

print("\nTest dataset disimpan ke:")
print(TEST_DATASET_PATH)

Dataset ML penuh disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\10_ml_dataset.csv

Train dataset disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\10_train_dataset.csv

Test dataset disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\10_test_dataset.csv


# Cell 16 — Simpan daftar fitur terpilih

In [16]:
selected_features_df = pd.DataFrame({
    "feature_name": selected_features
})

SELECTED_FEATURES_PATH = OUTPUT_TABLES_DIR / "10_selected_features.csv"
selected_features_df.to_csv(SELECTED_FEATURES_PATH, index=False)

print("Daftar fitur terpilih disimpan ke:")
print(SELECTED_FEATURES_PATH)

print("\nJumlah fitur terpilih:", len(selected_features))

Daftar fitur terpilih disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\10_selected_features.csv

Jumlah fitur terpilih: 47
